# Trích xuất Khung xương (BlazePose) - Kaggle

- Dùng Kaggle Dataset làm Input, xuất ra /kaggle/working và nén zip.

In [ ]:
# Cell 1: Clone mã nguồn
import os
REPO   = "https://github.com/tuan8p/Skeleton-EAA-Pose.git"
BRANCH = "dai"
WORKDIR = "/kaggle/working/Skeleton-EAA-Pose"
if not os.path.isdir(WORKDIR):
    !git clone -b {BRANCH} {REPO} {WORKDIR}
else:
    !git -C {WORKDIR} pull origin {BRANCH}
%cd {WORKDIR}

In [ ]:
# Cell 2: Cài đặt thư viện
!pip install -q mediapipe opencv-python numpy scipy pyyaml tqdm psutil

In [ ]:
# Cell 3: Tải Model BlazePose (Full)
import os, urllib.request
os.makedirs("models", exist_ok=True)
MODEL = "models/pose_landmarker_full.task"
if not os.path.exists(MODEL):
    urllib.request.urlretrieve(
        "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task",
        MODEL)
print("[OK] BlazePose model ready:", os.path.getsize(MODEL) // 1024, "KB")

In [ ]:
# Cell 4: Cấu hình thư mục & Parameter BlazePose
import os
from src.config_manager import ConfigManager

DATASET = "TSU"   # Chọn "PKU" hoặc "TSU"

DETECTION_OUTPUT_DIR = "/kaggle/input/datasets/tuan8p/bboxes-detection"
POSE_OUTPUT_DIR = "/kaggle/working/outputs_pose"

PKU_PATHS = {
    "video_dir":      "/kaggle/input/datasets/tuan8p/pku-rgb",
    "annotation_dir": "/kaggle/input/datasets/tuan8p/pku-annotation",
}

TSU_PATHS = {
    "video_dir":      "/kaggle/input/datasets/tuan8p/tsu-rgb/Videos_mp4",
    "annotation_dir": "/kaggle/input/datasets/tuan8p/tsu-annotation/Annotation_v1.0",
}

# --- Tự động chọn thư mục tương ứng --- 
if DATASET == "PKU":
    VIDEO_INPUT_DIR = PKU_PATHS["video_dir"]
    ANN_INPUT_DIR   = PKU_PATHS["annotation_dir"]
else:
    VIDEO_INPUT_DIR = TSU_PATHS["video_dir"]
    ANN_INPUT_DIR   = TSU_PATHS["annotation_dir"]

os.makedirs(POSE_OUTPUT_DIR, exist_ok=True)

# ==========================================
# CẤU HÌNH PIPELINE (Chỉnh sửa ở đây)
# ==========================================
SETTINGS = {
    # Các tuỳ chọn loại mô hình: pose_landmarker_lite.task, _full.task, _heavy.task
    "mediapipe.model_path": "models/pose_landmarker_heavy.task", 
    
    # Các ngưỡng tự tin của BlazePose (Tăng lên nếu muốn bỏ qua nhiễu)
    "mediapipe.min_detection_confidence": 0.5,
    "mediapipe.min_presence_confidence": 0.5,
    "mediapipe.min_tracking_confidence": 0.5,
    
    # Cơ chế Fallback và Nội suy (Temporal)
    "temporal.bbox_interp_max_gap": 10,     # Nội suy tối đa 6 frame nếu mất Bbox
    "temporal.empty_run_frames": 30,       # Nếu mất quá 30 frame (1s), coi là đi ra ngoài
    
    # Đầu ra toạ độ
    "output.coordinate_mode": "world",     # "world" (3D thực tế) hoặc "pixel"
    
    # Hệ thống chạy
    "runtime.num_workers": 4,
}

cfg = ConfigManager("config.yaml")
cfg.set("dataset", DATASET)
cfg.set("paths.video_dir", VIDEO_INPUT_DIR)
cfg.set("paths.annotation_dir", ANN_INPUT_DIR)
cfg.set("detection.output_dir", DETECTION_OUTPUT_DIR)
cfg.set("paths.output_dir", POSE_OUTPUT_DIR)

# Nạp Setting
for k, v in SETTINGS.items():
    cfg.set(k, v)

cfg.save("config.pose_runtime.yaml")
print(f"[OK] Đã lưu cấu hình!")

In [ ]:
# Cell 5: Chạy trích xuất khung xương (Đa luồng CPU Kaggle)
import os
from src.pipeline_orchestrator import PipelineOrchestrator
import concurrent.futures
from tqdm.notebook import tqdm

orch = PipelineOrchestrator(cfg=cfg)
videos = orch.reader.list_videos()
print(f"Tổng số video cần xử lý: {len(videos)}")

# Chia cắt danh sách video cho các tài khoản:
START, END = 0, 50
videos = videos[START:END]
print(f"Tài khoản này sẽ chạy {len(videos)} video (Từ index {START} đến {END-1}).")

NUM_THREADS = int(cfg.get("runtime.num_workers", 4))
print(f"Bắt đầu bóc xương với {NUM_THREADS} luồng song song...")

def process_single_video(vname):
    if orch.progress.is_video_done(vname):
        return f"Bỏ qua {vname} (Đã xong)"
    try:
        stats = orch.process_video(vname)
        orch.progress.mark_video_done(vname, stats.to_dict())
        failed = stats.total_frames - stats.ok_frames
        return f"Xong {vname} | Thành công: {stats.ok_frames}/{stats.total_frames} | Fail: {failed}"
    except Exception as e:
        return f"Lỗi {vname}: {str(e)}"

with concurrent.futures.ThreadPoolExecutor(max_workers=NUM_THREADS) as executor:
    futures = {executor.submit(process_single_video, v): v for v in videos}
    for future in tqdm(concurrent.futures.as_completed(futures), total=len(videos)):
        print(future.result())
print("[HOÀN TẤT] Quá trình bóc xương kết thúc!")

In [ ]:
# Cell 6: Nén kết quả thành file zip để tải về
import shutil
import os

zip_name = "/kaggle/working/pose_results_tsu"
print(f"Đang nén thư mục {POSE_OUTPUT_DIR}...")
shutil.make_archive(zip_name, 'zip', POSE_OUTPUT_DIR)
print(f"[OK] Đã tạo file: {zip_name}.zip ({os.path.getsize(zip_name+'.zip') // (1024*1024)} MB)")
print("-> BẠN CÓ THỂ TẢI TỪ CỘT BÊN PHẢI (Mục Output).")